# rl-crypto — quick smoke test

Offline checks of the public surface. Unlike `unified.ipynb`, nothing here
touches the network: it builds the models directly and asserts the
contracts (stable keys, `None`-not-fabricated, `interval_seconds`), so it
runs in a second and can be rerun freely.

In [ ]:
from rl_crypto import (
    SCHEMA_VERSION,
    Config,
    Dataset,
    DatasetRow,
    UnifiedRecord,
    __version__,
    interval_seconds,
)

print("rl-crypto", __version__)
print("schema   ", SCHEMA_VERSION)
assert __version__
assert SCHEMA_VERSION.startswith("rl-crypto/")

In [ ]:
record = UnifiedRecord(
    ticker="ETH_USD",
    query="ETH/USD",
    resolved_symbol="ETH/USD",
    resolved_name="Ethereum",
    price=2500.0,
    consensus_price=2501.5,
    market_ok=True,
    latest_sentiment=0.42,
    news_ok=True,
    notes=("news: live feed unavailable",),
)
payload = record.to_dict()
assert payload["schema"] == SCHEMA_VERSION
assert payload["market"]["price"] == 2500.0
assert payload["news"]["latest_sentiment"] == 0.42
assert payload["notes"] == ["news: live feed unavailable"]  # tuple -> list
print(record.to_json(indent=None)[:120])

In [ ]:
row = DatasetRow(
    ticker="ETH_USD",
    timestamp="2026-01-01T00:00:00Z",
    close=2500.0,
    open=2480.0,
    high=2520.0,
    low=2475.0,
    volume=12345.0,
    sentiment_score=0.42,
    article_count=3,
    novelty_flag=True,
    has_price=True,
    has_news=True,
)
dataset = Dataset(
    ticker="ETH_USD",
    interval="1h",
    start="2026-01-01",
    end="2026-01-02",
    rows=(row,),
)
assert dataset.joined_rows == 1
assert dataset.price_rows == 1 and dataset.news_rows == 1
assert dataset.to_dict()["coverage"]["rows"] == 1
print(dataset.to_json(indent=2)[:200])

In [ ]:
assert interval_seconds("1h") == 3600
assert interval_seconds("1d") == 86400
assert interval_seconds("nonsense") == 3600  # unknown -> one-hour floor

cfg = Config.from_env({})
assert cfg.lookback_hours == 1 and cfg.timeframe == "1d"
overridden = cfg.with_overrides(timeframe="1h")
assert overridden.timeframe == "1h" and cfg.timeframe == "1d"  # frozen

print("all checks passed")